# F02-P3 Threat

**Ecosystem Threat: Observed Disturbance, Ecosystem Pressure and Hazards, and Business-as-Usual Projection**

This section identifies current and potential threats within the project area, including human-induced disturbance, natural disturbance, and climate-related hazards. F02-P3 Threat addresses two questions: **“What are the potential threats?”** and **“What may happen if these threats are not mitigated?”**

> **Not runnable yet.** The analysis logic below is implemented in Python, but file access is currently stubbed.

**Status.** Complete as scoped for the **All Ecosystems** section.

**Known limitation.** Areas exposed to high disaster risk may affect the feasibility and effectiveness of NbS interventions. High-risk areas should therefore be carefully considered when selecting intervention locations.

## Setup

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio

from pyproj import Geod
from rasterio.mask import raster_geometry_mask
from rasterio.mask import mask
from scipy.ndimage import distance_transform_edt

---
## 7.1 Historical Deforestation (2014 to 2024)

Reports how much forest the AOI lost between 2014 and 2024 and the annual rate using Puyravaud.

**Data.** `FC2014.tif` (binary forest cover 2014, Tier 1-2) and `LC2024.tif` (20 class land cover
2024). Both dates use the same Tier 1-2 forest definition, through `forest_mask_2014` and
`forest_mask_2024` in `common.py`.

**Decisions locked.**

- `A2 = A1 - loss_ha`, gross loss of 2014 forest. Forest gain on non forest 2014 land is
  excluded, so the rate stays consistent with the reported loss.
- No comparison against a national rate. The component reports the site on its own terms. The
  national lookup, its CSV and the similar band threshold have been removed from the tool.
- The rate is shown to one decimal. Screening precision, not a monitoring figure.

**Puyravaud (2003).** `rate = (1 / (t2 - t1)) * ln(A2 / A1) * 100`

**Two edge cases decided here, not in the spec.**

1. No loss at all. The Puyravaud formula returns 0, which is correct but reads oddly as "an
   average of 0.0% per year", so the component uses a dedicated sentence instead.
2. Total loss, `A2 == 0`. All 2014 forest is gone and `ln(0)` is undefined. The component
   reports `rate_pct = None` and says the area lost all of its forest, rather than emitting an
   infinite rate.

**Example render.**

> Between 2014 and 2024, the project area lost 180 ha of forest, an average of 1.6% per year.

**Downstream use.** Observed loss is the empirical basis of the trajectory (Axis 2) and a direct
threat intensity signal. The AUD pathway signal now comes from the modelled risk in 1.6 rather
than from a national rate comparison here.

In [ ]:
def analyze_historical_deforestation(aoi: AOI) -> ComponentResult:
    """Component 1.5. Forest loss 2014 to 2024 and its annual rate."""
    f2014 = forest_mask_2014(aoi)
    f2024 = forest_mask_2024(aoi)

    if f2014.is_empty:
        return not_applicable(
            "1.5 Historical Deforestation",
            "No forest was present in this project area in 2014, so a deforestation rate "
            "cannot be calculated.",
        )

    # Derived output rasters: forest at each date and the gross-loss grid.
    _loss = f2014.mask & ~f2024.mask
    loss_slice = RasterSlice(
        np.ma.masked_array(_loss.astype("float32"), mask=False),
        f2014.pixel_area_ha, f2014.transform, f2014.crs,
    )
    forest_rasters = {
        "1.5_forest_2014": f2014.as_slice(),
        "1.5_forest_2024": f2024.as_slice(),
        "1.5_forest_loss": loss_slice,
    }

    a1 = f2014.area_ha
    # Gross loss of 2014 forest. Gain elsewhere is deliberately not netted off.
    loss_ha = int((f2014.mask & ~f2024.mask).sum()) * f2014.pixel_area_ha
    a2 = a1 - loss_ha

    if loss_ha <= 0:
        rate_pct = 0.0
        narrative = "No forest loss was detected in this project area between 2014 and 2024."
    elif a2 <= 0:
        rate_pct = None  # ln(0) guard, see the note above
        narrative = (
            f"Between 2014 and 2024, the project area lost all of its {fmt_ha(a1)} of forest."
        )
    else:
        rate_pct = abs((1.0 / DEFOR_PERIOD_YEARS) * math.log(a2 / a1) * 100.0)
        narrative = (
            f"Between 2014 and 2024, the project area lost {fmt_ha(loss_ha)} of forest, an "
            f"average of {rate_pct:.1f}% per year."
        )

    return ComponentResult(
        component="1.5 Historical Deforestation",
        applicable=True,
        narrative=narrative,
        tables={},
        values={
            "forest_2014_ha": a1,
            "forest_2024_ha": a2,
            "loss_ha": loss_ha,
            "rate_pct": rate_pct,
        },
        rasters=forest_rasters,
    )


results["1.5"] = analyze_historical_deforestation(aoi)
show_result(results["1.5"])

---
## 7.2 Historical Burned Area

Reports how much of the AOI has burned over the GABAM record, 2014 to 2024, as an annual series
(bar chart) and one burned footprint.

**Data.** GABAM annual burned maps (`GABAM_RASTER_TEMPLATE`), one binary raster per year, value
1 = burned and 0 = nodata, about 30 m, EPSG:4326. Eleven years, 2014 to 2024.

**Two totals, on purpose.** A pixel can burn in more than one year, so:

- The headline is the **union**: the area that burned at least once. It is always at most the AOI
  area and reads as the true burned footprint.
- The bar chart is **per year**: a pixel that reburns is counted in each year it burns, which is
  what an annual series should show.
- The **sum** of the annual areas is also kept in `values` (`burned_sum_ha`). It can exceed the
  AOI area and measures burn events, not footprint, so it must not be read as an area.

**Decisions locked.**

- Burned = the valid (non nodata) pixels, since GABAM encodes 0 as nodata.
- All years are aligned to the first year's grid (`like=`), so the union is an exact pixel-wise OR.
- Nearest resampling only, never interpolate a burned or not-burned flag.

**Example render.**

> Between 2014 and 2024, 1,240 ha of the project area burned at least once, about 8.6% of the area.

**Downstream use.** A fire history raises permanence risk and points to fire-driven degradation;
it also supports fire-management activities in the pathway logic.

In [ ]:
@dataclass(frozen=True)
class BurnedYear:
    year: int
    burned_ha: float


def analyze_historical_burned_area(aoi: AOI) -> ComponentResult:
    """Component 3.7. Annual burned area from GABAM, 2014 to 2024, and the burned footprint.

    GABAM is a binary annual burned map (1 = burned, 0 = nodata). A pixel can burn in several
    years, so two totals are reported: the union (area burned at least once), used as the
    headline, and the sum of the annual areas, kept in values. The bar chart is per year, where a
    reburn is counted in each year it happens.
    """
    ref = None                    # first year fixes the grid; later years align to it via like=
    union = None                  # boolean, burned at least once, on the ref grid
    annual: list[BurnedYear] = []
    summed_ha = 0.0

    for year in GABAM_YEARS:
        path = GABAM_RASTER_TEMPLATE.format(year=year)
        raster = load_raster_clipped(path, aoi, resampling="nearest", like=ref)
        if ref is None:
            ref = raster
        burned = ~np.ma.getmaskarray(raster.values)   # unmasked = burned inside the AOI
        burned_ha = float(burned.sum()) * raster.pixel_area_ha
        annual.append(BurnedYear(year=year, burned_ha=burned_ha))
        summed_ha += burned_ha
        union = burned.copy() if union is None else (union | burned)

    union_ha = float(union.sum()) * ref.pixel_area_ha if union is not None else 0.0
    union_pct = (union_ha / aoi.area_ha * 100.0) if aoi.area_ha > 0 else 0.0

    if union_ha <= 0:
        narrative = "No burned area was detected in this project area between 2014 and 2024."
    else:
        narrative = (
            f"Between 2014 and 2024, {fmt_ha(union_ha)} of the project area burned at least once, "
            f"about {union_pct:.1f}% of the area."
        )

    return ComponentResult(
        component="3.7 Historical Burned Area",
        applicable=True,
        narrative=narrative,
        tables={"annual_burned_area": annual},
        values={
            "burned_union_ha": union_ha,
            "burned_union_pct": union_pct,
            "burned_sum_ha": summed_ha,
            "years": [b.year for b in annual],
        },
        rasters={},
    )


results["3.7"] = analyze_historical_burned_area(aoi)
show_result(results["3.7"])

---
## 3.3 Mangrove Ecosystem

Reports the drivers of mangrove disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.** Post-processing steps identified where fire coincided with the clearing of agricultural land, based on the presence of VIIRS fire alerts (within a 500 m buffer around the alert) and a low post-disturbance normalized-burn ration in the following month’s Sentinen-2 composite.
Post-processing steps masking out the area outside forest_disturbance_v3.tif and re-calibrate
with disaster risk from ADPC. However, this data only consider high and very high disaster risk
as part of forest disturbance.

## Mangrove Disturbance

| Indicator | Value |
|---|---:|
| **Total area** | **100,810.15 ha** |
| **Remaining mangrove forest** | **55,491.97 ha (55.05%)** |
| **Disturbed** | **150.89 ha (0.15%)** |
| **Main pressure** | **Not identified** |

### Mangrove Disturbance Drivers

#### Non-natural Drivers

None detected.

#### Natural Drivers

None detected.

#### Other Drivers

- Other

In [ ]:

aoi = gpd.read_file(
    USER_AOI
)

if aoi.empty:
    raise ValueError(
        "AOI contains no features."
    )

if aoi.crs is None:
    raise ValueError(
        "AOI has no CRS."
    )

aoi = aoi[
    aoi.geometry.notna()
    & ~aoi.geometry.is_empty
].copy()

if aoi.empty:
    raise ValueError(
        "AOI contains no valid geometry."
    )


# =============================================================================
# READ RASTER CLIPPED TO USER AOI
# =============================================================================

def read_raster(
    raster_path
):

    with rasterio.open(
        raster_path
    ) as src:

        if src.crs is None:
            raise ValueError(
                f"Raster has no CRS: {raster_path}"
            )

        polygon = (
            aoi.to_crs(src.crs)
            if aoi.crs != src.crs
            else aoi
        )

        geometries = [
            geom.__geo_interface__
            for geom in polygon.geometry
        ]

        data, transform = mask(
            src,
            geometries,
            crop=True,
            filled=False
        )

        return (
            data[0],
            transform,
            src.crs
        )


# =============================================================================
# CHECK GRID ALIGNMENT
# =============================================================================

def check_grid(
    reference,
    reference_transform,
    data,
    transform,
    name
):

    if reference.shape != data.shape:

        raise ValueError(
            f"Grid size mismatch: {name}"
        )

    if not np.allclose(
        reference_transform,
        transform
    ):

        raise ValueError(
            f"Grid alignment mismatch: {name}"
        )


# =============================================================================
# AREA CALCULATION
# EPSG:4326 -> geodesic hectare
# =============================================================================

GEOD = Geod(
    ellps="WGS84"
)


def area_ha(
    binary_mask,
    transform
):

    # Avoid MaskError from masked arrays
    binary_mask = np.ma.filled(
        binary_mask,
        False
    ).astype(bool)

    total_m2 = 0.0

    pixel_width = abs(
        transform.a
    )

    for row in range(
        binary_mask.shape[0]
    ):

        pixel_count = np.count_nonzero(
            binary_mask[row]
        )

        if pixel_count == 0:
            continue

        north = (
            transform.f
            + row * transform.e
        )

        south = (
            north
            + transform.e
        )

        west = transform.c
        east = west + pixel_width

        pixel_area_m2, _ = (
            GEOD.polygon_area_perimeter(
                [west, east, east, west],
                [north, north, south, south]
            )
        )

        total_m2 += (
            pixel_count
            * abs(pixel_area_m2)
        )

    return (
        total_m2 / 10000
    )


# =============================================================================
# LOAD RASTERS
# =============================================================================

ecosystem, transform, crs = (
    read_raster(
        ECOSYSTEM
    )
)

historical, historical_transform, _ = (
    read_raster(
        HISTORICAL
    )
)

forest2024, forest2024_transform, _ = (
    read_raster(
        FOREST_2024
    )
)

disturbance, disturbance_transform, _ = (
    read_raster(
        DISTURBANCE
    )
)

drivers, drivers_transform, _ = (
    read_raster(
        DRIVERS
    )
)

storm, storm_transform, _ = (
    read_raster(
        STORM_RISK
    )
)


# =============================================================================
# VALIDATE GRIDS
# =============================================================================

check_grid(
    ecosystem,
    transform,
    historical,
    historical_transform,
    "historical_deforestation_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    forest2024,
    forest2024_transform,
    "forest_2024_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    disturbance,
    disturbance_transform,
    "forest_disturbance_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    drivers,
    drivers_transform,
    "drivers_disturbance_v3.tif"
)

check_grid(
    ecosystem,
    transform,
    storm,
    storm_transform,
    "risk_storm_v3.tif"
)


# =============================================================================
# VALID PIXELS
# =============================================================================

ecosystem_valid = (
    ~np.ma.getmaskarray(
        ecosystem
    )
)

historical_valid = (
    ~np.ma.getmaskarray(
        historical
    )
)

forest2024_valid = (
    ~np.ma.getmaskarray(
        forest2024
    )
)

disturbance_valid = (
    ~np.ma.getmaskarray(
        disturbance
    )
)

drivers_valid = (
    ~np.ma.getmaskarray(
        drivers
    )
)

storm_valid = (
    ~np.ma.getmaskarray(
        storm
    )
)


# =============================================================================
# 1. TOTAL MANGROVE AREA
#
# User AOI
# -> ecosystem_v3 == 2
# =============================================================================

mangrove_mask = (
    ecosystem_valid
    & (
        ecosystem.data
        == MANGROVE_CLASS
    )
)

total_mangrove_area = area_ha(
    mangrove_mask,
    transform
)


# =============================================================================
# 2. REMAINING MANGROVE FOREST
#
# Mangrove mask
# -> historical_deforestation_v3 == 1
# =============================================================================

remaining_mask = (
    mangrove_mask
    & historical_valid
    & (
        historical.data
        == REMAINING_FOREST_CLASS
    )
)

remaining_area = area_ha(
    remaining_mask,
    transform
)


# =============================================================================
# 3. CURRENT MANGROVE FOREST
#
# Mangrove mask
# -> forest_2024_v3 == 1
# =============================================================================

current_mangrove_mask = (
    mangrove_mask
    & forest2024_valid
    & (
        forest2024.data
        == CURRENT_FOREST_CLASS
    )
)


# =============================================================================
# 4. DISTURBED MANGROVE
#
# Current mangrove
# -> forest_disturbance_v3 > 0
#
# This becomes the MASTER MASK for all driver analysis.
# =============================================================================

disturbed_mangrove_mask = (
    current_mangrove_mask
    & disturbance_valid
    & (
        disturbance.data
        > DISTURBANCE_THRESHOLD
    )
)

disturbed_area = area_ha(
    disturbed_mangrove_mask,
    transform
)


# =============================================================================
# PERCENTAGES
# =============================================================================

remaining_percentage = (
    remaining_area
    / total_mangrove_area
    * 100
    if total_mangrove_area > 0
    else 0
)

disturbed_percentage = (
    disturbed_area
    / total_mangrove_area
    * 100
    if total_mangrove_area > 0
    else 0
)


# =============================================================================
# 5. NON-NATURAL DRIVERS
#
# Disturbed mangrove
# -> drivers_disturbance_v3
#
# 1,2,3,4 = Commodities
# 5       = Settlement
# =============================================================================

commodities_mask = (
    disturbed_mangrove_mask
    & drivers_valid
    & np.isin(
        drivers.data,
        COMMODITY_CLASSES
    )
)

settlement_mask = (
    disturbed_mangrove_mask
    & drivers_valid
    & (
        drivers.data
        == SETTLEMENT_CLASS
    )
)


commodities_area = area_ha(
    commodities_mask,
    transform
)

settlement_area = area_ha(
    settlement_mask,
    transform
)


non_natural_drivers = []

if np.any(
    commodities_mask
):

    non_natural_drivers.append(
        "Commodities"
    )

if np.any(
    settlement_mask
):

    non_natural_drivers.append(
        "Settlement"
    )


# =============================================================================
# 6. MAIN PRESSURE
#
# Largest overlap area between:
# - Commodities
# - Settlement
# =============================================================================

if commodities_area > settlement_area:

    main_pressure = (
        "Commodities"
    )

elif settlement_area > commodities_area:

    main_pressure = (
        "Settlement"
    )

elif (
    commodities_area > 0
    and settlement_area > 0
):

    main_pressure = (
        "Commodities and Settlement"
    )

else:

    main_pressure = (
        "Not identified"
    )


# =============================================================================
# 7. NATURAL DRIVER
#
# Disturbed mangrove
# -> risk_storm_v3
# -> pixel 4 or 5
# =============================================================================

storm_mask = (
    disturbed_mangrove_mask
    & storm_valid
    & np.isin(
        storm.data,
        STORM_RISK_CLASSES
    )
)


natural_drivers = []

if np.any(
    storm_mask
):

    natural_drivers.append(
        "Extreme climate event"
    )


# =============================================================================
# 8. OTHER DRIVER
#
# Disturbed mangrove pixels that DO NOT overlap
# known drivers_disturbance_v3 classes 1-5.
#
# Natural storm risk is independent and does not remove
# a pixel from the "Other" classification.
# =============================================================================

known_driver_mask = (
    disturbed_mangrove_mask
    & drivers_valid
    & np.isin(
        drivers.data,
        [1, 2, 3, 4, 5]
    )
)

other_mask = (
    disturbed_mangrove_mask
    & ~known_driver_mask
)


other_drivers = []

if np.any(
    other_mask
):

    other_drivers.append(
        "Other"
    )


# =============================================================================
# DEBUG: DATA INTERACTION
#
# Useful during testing to identify where pixels disappear.
# =============================================================================

print()
print("=" * 75)
print("MANGROVE MASK CHECK")
print("=" * 75)

print(
    f"Mangrove pixels             : "
    f"{np.count_nonzero(mangrove_mask):,}"
)

print(
    f"Current mangrove pixels     : "
    f"{np.count_nonzero(current_mangrove_mask):,}"
)

print(
    f"Disturbed mangrove pixels   : "
    f"{np.count_nonzero(disturbed_mangrove_mask):,}"
)

print(
    f"Commodities pixels          : "
    f"{np.count_nonzero(commodities_mask):,}"
)

print(
    f"Settlement pixels           : "
    f"{np.count_nonzero(settlement_mask):,}"
)

print(
    f"Storm risk 4/5 pixels       : "
    f"{np.count_nonzero(storm_mask):,}"
)

print(
    f"Other pixels                : "
    f"{np.count_nonzero(other_mask):,}"
)

# =============================================================================
# BACKEND RESULT
# =============================================================================

result = {

    "mangrove": {

        "total_area_ha":
            round(
                total_mangrove_area,
                2
            ),

        "remaining_forest": {

            "area_ha":
                round(
                    remaining_area,
                    2
                ),

            "percentage":
                round(
                    remaining_percentage,
                    2
                )
        },

        "disturbed": {

            "area_ha":
                round(
                    disturbed_area,
                    2
                ),

            "percentage":
                round(
                    disturbed_percentage,
                    2
                )
        },

        "main_pressure":
            main_pressure,

        "drivers": {

            "non_natural":
                non_natural_drivers,

            "natural":
                natural_drivers,

            "other":
                other_drivers
        }
    }
}


# =============================================================================
# FINAL OUTPUT
# =============================================================================

print()
print("=" * 75)
print("MANGROVE DISTURBANCE")
print("=" * 75)

print(
    f"Total area                : "
    f"{result['mangrove']['total_area_ha']:,.2f} ha"
)

print(
    f"Remaining mangrove forest : "
    f"{result['mangrove']['remaining_forest']['area_ha']:,.2f} ha "
    f"({result['mangrove']['remaining_forest']['percentage']:.2f}%)"
)

print(
    f"Disturbed                 : "
    f"{result['mangrove']['disturbed']['area_ha']:,.2f} ha "
    f"({result['mangrove']['disturbed']['percentage']:.2f}%)"
)

print(
    f"Main pressure             : "
    f"{result['mangrove']['main_pressure']}"
)


print()
print("Mangrove disturbance drivers")
print("-" * 75)


print()
print("Non-natural drivers:")

if non_natural_drivers:

    for driver in non_natural_drivers:

        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )


print()
print("Natural drivers:")

if natural_drivers:

    for driver in natural_drivers:

        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )


print()
print("Other drivers:")

if other_drivers:

    for driver in other_drivers:

        print(
            f"  - {driver}"
        )

else:

    print(
        "  None detected"
    )

## 7.4 Peatland Ecosystem

Reports the drivers of peatland disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.**  
The drainage canal dataset developed by Dadap et al. (2021) captures drainage canals located across both peatland and non-peatland areas. Therefore, the dataset was masked using `ecosystem_v3.tif` to retain only drainage canals occurring within mapped peatland areas in Southeast Asia.

Drainage pressure, however, refers to the proximity of drainage canals to the project area, including canals located outside the project boundary that may still influence peatland conditions within the project area. This follows previous findings indicating that:

a) Astiani et al. (2017) found that canals can influence water-table depth up to 500 m from the surrounding area.

b) Wedeux et al. (2020) found that canal networks may affect forest biomass growth up to 1 km away.

## Peatland Disturbance

| Indicator | Value |
|---|---:|
| **Total area** | **24,694.84 ha** |
| **Remaining peatland forest** | **1,182.02 ha (4.79%)** |
| **Disturbed** | **165.43 ha (0.67%)** |
| **Converted / Loss** | **4,968.08 ha (20.12%)** |

### Peatland Disturbance Drivers

| Driver | Status |
|---|---:|
| **Canal proximity** | **High** |
| **Nearest canal** | **0.00 m** |
| **Drainage pressure** | **High** |
| **Fire risk** | **High** |

In [ ]:
def read_raster(
    path,
    reference=None,
    reference_transform=None,
    reference_crs=None
):

    with rasterio.open(path) as src:

        polygon = (
            aoi.to_crs(src.crs)
            if aoi.crs != src.crs
            else aoi
        )

        try:

            data, raster_transform = mask(
                src,
                polygon.geometry,
                crop=True,
                filled=False
            )

            return (
                data[0],
                raster_transform,
                src.crs
            )

        except ValueError as e:

            if (
                "Input shapes do not overlap raster" not in str(e)
                or reference is None
            ):
                raise

            # No raster coverage inside AOI → return zero
            data = np.ma.array(
                np.zeros(
                    reference.shape,
                    dtype="float32"
                ),
                mask=False
            )

            return (
                data,
                reference_transform,
                reference_crs
            )

# =============================================================================
# AREA CALCULATION
# EPSG:4326 -> hectares
# =============================================================================

GEOD = Geod(
    ellps="WGS84"
)


def area_ha(
    binary_mask,
    transform
):

    binary_mask = np.asarray(
        binary_mask,
        dtype=bool
    )

    total_m2 = 0.0

    pixel_width = abs(
        transform.a
    )

    for row in range(
        binary_mask.shape[0]
    ):

        count = np.count_nonzero(
            binary_mask[row]
        )

        if count == 0:
            continue

        north = (
            transform.f
            + row * transform.e
        )

        south = (
            north
            + transform.e
        )

        west = transform.c
        east = west + pixel_width

        pixel_m2, _ = (
            GEOD.polygon_area_perimeter(
                [west, east, east, west],
                [north, north, south, south]
            )
        )

        total_m2 += (
            count
            * abs(pixel_m2)
        )

    return (
        total_m2 / 10000
    )


# =============================================================================
# LOAD RASTERS
# =============================================================================

ecosystem, transform, crs = read_raster(
    ECOSYSTEM
)

historical, historical_transform, _ = read_raster(
    HISTORICAL,
    ecosystem,
    transform,
    crs
)

forest2024, forest2024_transform, _ = read_raster(
    FOREST_2024,
    ecosystem,
    transform,
    crs
)

disturbance, disturbance_transform, _ = read_raster(
    DISTURBANCE,
    ecosystem,
    transform,
    crs
)

canal_density, canal_transform, _ = read_raster(
    PEAT_CANALS_DENSITY,
    ecosystem,
    transform,
    crs
)

fire_risk, fire_transform, _ = read_raster(
    FIRE_RISK,
    ecosystem,
    transform,
    crs
)

drainage_canals, drainage_transform, _ = read_raster(
    DRAINAGE_CANALS,
    ecosystem,
    transform,
    crs
)

# =============================================================================
# 1. TOTAL PEATLAND
#
# AOI
# -> ecosystem_v3 == 3
# =============================================================================

peatland_mask = (
    ecosystem == PEATLAND
)

total_area = area_ha(
    peatland_mask,
    transform
)


# =============================================================================
# 2. REMAINING PEATLAND FOREST
#
# Peatland
# -> historical_deforestation_v3 == 1
# =============================================================================

remaining_mask = (
    peatland_mask
    & (
        historical
        == REMAINING_FOREST
    )
)

remaining_area = area_ha(
    remaining_mask,
    transform
)


# =============================================================================
# 3. CURRENT PEATLAND FOREST
#
# Peatland
# -> forest_2024_v3 == 1
# =============================================================================

current_peatland_mask = (
    peatland_mask
    & (
        forest2024
        == CURRENT_FOREST
    )
)


# =============================================================================
# 4. DISTURBED PEATLAND
#
# Current peatland forest
# -> forest_disturbance_v3 > 0
# =============================================================================

disturbed_mask = (
    current_peatland_mask
    & (
        disturbance
        > DISTURBANCE_THRESHOLD
    )
)

disturbed_area = area_ha(
    disturbed_mask,
    transform
)


# =============================================================================
# 5. CONVERTED / LOSS
#
# Peatland
# -> historical_deforestation_v3 == 2
# =============================================================================

loss_mask = (
    peatland_mask
    & (
        historical
        == FOREST_LOSS
    )
)

loss_area = area_ha(
    loss_mask,
    transform
)


# =============================================================================
# PERCENTAGES
# =============================================================================

def percentage(area):

    if total_area == 0:
        return 0

    return (
        area
        / total_area
        * 100
    )


remaining_percentage = percentage(
    remaining_area
)

disturbed_percentage = percentage(
    disturbed_area
)

loss_percentage = percentage(
    loss_area
)


# =============================================================================
# CANAL PROXIMITY
#
# drainage_canal_v3.tif
# pixel 1 = drainage canal
#
# Current peatland forest -> nearest canal
#
# <= 500 m       = High
# > 500-1000 m   = Medium
# > 1000 m       = Low
# =============================================================================

from rasterio.features import shapes
from shapely.geometry import shape


def mask_to_geometry(
    binary_mask,
    transform,
    crs
):

    geometries = [
        shape(geom)
        for geom, value in shapes(
            binary_mask.astype(
                np.uint8
            ),
            mask=binary_mask,
            transform=transform
        )
        if value == 1
    ]

    if not geometries:
        return None

    return (
        gpd.GeoSeries(
            geometries,
            crs=crs
        )
        .union_all()
    )


def canal_proximity_level():

    # Current peatland forest
    if not np.any(
        current_peatland_mask
    ):
        return "Not identified", None


    # Drainage canal pixel = 1
    canal_mask = (
        drainage_canals == 1
    )


    if not np.any(
        canal_mask
    ):
        return "Low", None


    # Convert raster masks to geometry
    peat_forest_geometry = mask_to_geometry(
        current_peatland_mask,
        transform,
        raster_crs
    )

    canal_geometry = mask_to_geometry(
        canal_mask,
        drainage_transform,
        drainage_crs
    )


    if (
        peat_forest_geometry is None
        or canal_geometry is None
    ):
        return "Low", None


    # Use projected CRS for metre distance
    metric_crs = (
        aoi.estimate_utm_crs()
    )


    peat_metric = (
        gpd.GeoSeries(
            [peat_forest_geometry],
            crs=raster_crs
        )
        .to_crs(metric_crs)
        .iloc[0]
    )


    canal_metric = (
        gpd.GeoSeries(
            [canal_geometry],
            crs=drainage_crs
        )
        .to_crs(metric_crs)
        .iloc[0]
    )


    # Nearest distance
    distance_m = (
        peat_metric.distance(
            canal_metric
        )
    )


    if distance_m <= 500:

        level = "High"

    elif distance_m <= 1000:

        level = "Medium"

    else:

        level = "Low"


    return (
        level,
        float(distance_m)
    )


canal_proximity, canal_distance_m = (
    canal_proximity_level()
)
# =============================================================================
# 7. DRAINAGE PRESSURE
#
# Peatland
# -> peat_canals_density_v3
#
# 3 = High
# 2 = Medium
# 1 = Low
#
# UI uses highest class present.
# =============================================================================

drainage_values = np.unique(
    canal_density[
        peatland_mask
    ]
)


if 3 in drainage_values:

    drainage_pressure = (
        "High"
    )

elif 2 in drainage_values:

    drainage_pressure = (
        "Medium"
    )

elif 1 in drainage_values:

    drainage_pressure = (
        "Low"
    )

else:

    drainage_pressure = (
        "Not identified"
    )


# =============================================================================
# 8. FIRE RISK
#
# Peatland
# -> fire_risk_v3
#
# 4 = High
# 3 = Medium
# 2 = Low
# 1 = Very low
# 0 / NoData = No risk
#
# UI uses highest class present.
# =============================================================================

fire_values = np.unique(
    fire_risk[
        peatland_mask
    ]
)


if 4 in fire_values:

    fire_risk_level = (
        "High"
    )

elif 3 in fire_values:

    fire_risk_level = (
        "Medium"
    )

elif 2 in fire_values:

    fire_risk_level = (
        "Low"
    )

elif 1 in fire_values:

    fire_risk_level = (
        "Very low"
    )

else:

    fire_risk_level = (
        "No risk"
    )


# =============================================================================
# BACKEND RESULT
# =============================================================================

result = {

    "peatland": {

        "total_area_ha":
            round(
                total_area,
                2
            ),

        "remaining_forest": {

            "area_ha":
                round(
                    remaining_area,
                    2
                ),

            "percentage":
                round(
                    remaining_percentage,
                    2
                )
        },

        "disturbed": {

            "area_ha":
                round(
                    disturbed_area,
                    2
                ),

            "percentage":
                round(
                    disturbed_percentage,
                    2
                )
        },

        "converted_loss": {

            "area_ha":
                round(
                    loss_area,
                    2
                ),

            "percentage":
                round(
                    loss_percentage,
                    2
                )
        },

        "drivers": {

            "canal_proximity":
                canal_proximity,

            "canal_distance_m":
                (
                    round(
                        canal_distance_m,
                        2
                    )
                    if canal_distance_m
                    is not None
                    else None
                ),

            "drainage_pressure":
                drainage_pressure,

            "fire_risk":
                fire_risk_level
        }
    }
}


# =============================================================================
# FINAL OUTPUT
# =============================================================================

r = result[
    "peatland"
]


print()
print("=" * 75)
print("PEATLAND DISTURBANCE")
print("=" * 75)


print(
    f"Total area                : "
    f"{r['total_area_ha']:,.2f} ha"
)


print(
    f"Remaining peatland forest : "
    f"{r['remaining_forest']['area_ha']:,.2f} ha "
    f"({r['remaining_forest']['percentage']:.2f}%)"
)


print(
    f"Disturbed                 : "
    f"{r['disturbed']['area_ha']:,.2f} ha "
    f"({r['disturbed']['percentage']:.2f}%)"
)


print(
    f"Converted / Loss          : "
    f"{r['converted_loss']['area_ha']:,.2f} ha "
    f"({r['converted_loss']['percentage']:.2f}%)"
)


print()
print("Peatland disturbance drivers")
print("-" * 75)


print(
    f"Canal proximity  : "
    f"{r['drivers']['canal_proximity']}"
)


if (
    r["drivers"]["canal_distance_m"]
    is not None
):

    print(
        f"Nearest canal    : "
        f"{r['drivers']['canal_distance_m']:,.2f} m"
    )


print(
    f"Drainage pressure: "
    f"{r['drivers']['drainage_pressure']}"
)


print(
    f"Fire risk        : "
    f"{r['drivers']['fire_risk']}"
)

---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_GENERAL)
print(f"Saved {path}")